# Noh Implosion (1D)

This notebook runs the classical Noh implosion: uniform cold gas moving inward at a constant speed `v_s`, which converges on the origin and forms an outward-propagating shock. The exact post-shock density has a closed form, `rho_s = rho0 * ((gamma+1)/(gamma-1))^dim`, drawn as the reference line on the density panel; the shock fronts (`+/- v_s * t`) are drawn on the pressure panel.

Like `sod_1d.ipynb`, this notebook calls the real case code (`warpSPH.cases.noh.nohCase`) rather than re-deriving it, and keeps the step loop unrolled in a cell instead of hiding it inside `warpSPH.runner.run()`. Plotting calls `drawNoh` (the same per-frame redraw `nohCase.setupPlot`/`updatePlot` use internally, exported directly from `warpSPH.cases.noh` for this) rather than going through the `Case` hooks' `openWindow`/`pumpEvents`, which does not live-update reliably inside a Jupyter cell in this environment.

Precision note: switching between single and double precision is controlled in the import/configuration cell below. Because precision is set when core modules/kernels are initialized, any precision change requires a kernel restart before re-running the notebook.

![](outputs/04-Noh_Implosion.gif)


In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.noh import nohCase, shockState, drawNoh
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.io import createOutFile, prepExport, writeInitialData, writeFrame

import os
import matplotlib.pyplot as plt
import torch
from tqdm.autonotebook import tqdm


In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on
# `04-noh-implosion.py`, made explicit and editable here.
# `nohCase.defaults`/`nohCase.params` are the same values the CLI script
# starts from -- anything not overridden below just keeps its case default.
spec = CaseSpec(caseName=nohCase.name, scheme=nohCase.scheme,
                params=dict(nohCase.params)) \
    .merged(**nohCase.defaults)

spec = spec.merged(
    # --- discretisation ------------------------------------------------
    nx=200,
    dim=1,
    L=2.0,

    # --- time stepping -----------------------------------------------------
    tLimit=0.6,
    # `sampleNoh1D` does not set a CFL dt of its own, and `nohCase` has no
    # `timestep` hook either, so dt needs an explicit value and stays fixed.
    dt=1e-4,

    # --- output --------------------------------------------------------------
    plot=True, show=True, plotInterval=25,
    store=False,

    # --- Noh's own knobs -----------------------------------------------------
    params=dict(
        v_s=1 / 3, gamma=5 / 3, rho0=1.0,
    ),
)
spec


In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`nohCase.buildSystem` -> `sampleNoh1D`), not re-derived here.
ctx = buildContext(nohCase, spec)
nohCase.configureScheme(ctx)
system = nohCase.buildSystem(ctx)
runningState = system.initializeNewState()

rhoShock, pressureShock = shockState(ctx)
print(f"rho_s = {rhoShock}, P_s = {pressureShock}")


In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

# Direct drawNoh + plt.subplots(), not nohCase.setupPlot -- see the intro
# cell for why.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    fig, axis = plt.subplots(1, 3, figsize=(10, 5), squeeze=False)
    drawNoh(ctx, runningState, (fig, axis))
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

outFile = None
groups = None
if spec.store and spec.storeMode == 'trajectory':
    extraData = nohCase.extraData(ctx, runningState)
    outFile = createOutFile(ctx.exportPath)
    groups = writeInitialData(ctx.exportPath, outFile, ctx.scheme, ctx.config, ctx.schemeConfig,
                              spec, runningState, extraData=extraData, extraFields=nohCase.extraFields)


In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(spec.tLimit / dt)
storeSteps = max(1, int(spec.exportInterval / dt)) if spec.storeMode == 'trajectory' \
    else max(1, spec.storeInterval)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point ---------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -------------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = nohCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        drawNoh(ctx, runningState, (fig, axis))
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if outFile is not None and (i % storeSteps == 0 or i == nSteps - 1):
        writeFrame(groups, i, stepResult.state, stepResult.stages, config=ctx.config,
                  schemeConfig=ctx.schemeConfig, uniqueParticles=True, writeStages=False,
                  extraFields=nohCase.extraFields)


In [ ]:
if outFile is not None:
    outFile.close()

if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)
